# ANN Surrogate Model for FCC Catalyst Optimization

Trains an ANN surrogate model using catalyst/process variables to predict conversion and product distribution.


In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

# -----------------------------
# 1) Fixed 5-node "P&ID" graph
# -----------------------------
N_NODES = 5
edges = [(0,1),(1,2),(2,3),(3,4)]
A = np.zeros((N_NODES,N_NODES), dtype=np.float32)
for u,v in edges:
    A[u,v] = 1.0; A[v,u] = 1.0
A += np.eye(N_NODES, dtype=np.float32)  # self loops
An = A / (A.sum(axis=1, keepdims=True) + 1e-6)        # row-normalized adjacency
An_t = torch.tensor(An, dtype=torch.float32)

# node type one-hot: Feed, Acid, Acid, 13X-bed, Product
node_types = np.array([0,1,1,2,3], dtype=np.int64)
num_types = 4
type_onehot = np.eye(num_types, dtype=np.float32)[node_types]  # (5,4)
BASE = type_onehot.shape[1]

# -----------------------------------------
# 2) Toy "truth" function (13X dries water)
# -----------------------------------------
def clamp(x, lo, hi): return np.clip(x, lo, hi)

def truth(u):
    """
    u = [T1,P1,acid1,tau1,  T2,P2,acid2,tau2,  T3,P3,WHSV,H2, zfrac, mload, act,  Fkgph,O0,W0,V0]
    outputs = [Yield, Moisture, Oxygenates, Viscosity, Stability, Q_kW, dP_bar]
    """
    (T1,P1,a1,tau1, T2,P2,a2,tau2, T3,P3,WHSV,H2, zfrac,mload,act, Fkgph,O0,W0,V0) = u

    sev1 = (a1*tau1)*(T1/480.0)**0.6
    sev2 = (a2*tau2)*(T2/500.0)**0.6

    Oa = O0*(1-0.10*clamp(sev1,0,1.5))*(1-0.12*clamp(sev2,0,1.5))
    Wa = W0*(1-0.06*clamp(sev1,0,1.5))*(1-0.06*clamp(sev2,0,1.5))
    Va = V0*(1-0.04*clamp(sev1,0,1.5))*(1-0.05*clamp(sev2,0,1.5))
    Ya = 0.62*(1-0.01*clamp(sev1,0,2))*(1-0.01*clamp(sev2,0,2))

    sev3 = (T3/650.0)**1.2*(P3/30.0)**0.5*(H2**0.6)*act/(clamp(WHSV,0.2,10.0))

    # 13X effect: strong water adsorption (higher at lower T); modest oxygenate trapping
    ads_water = zfrac*1.2*(650.0/max(T3,1e-3))**0.7
    ads_oxyg  = zfrac*0.25

    # "modified 13X" proxy improves deoxygenation slightly (metal loading)
    mod_boost = 1 + 0.15*clamp(mload,0,1.0)

    Oout = Oa*(1-0.45*clamp(sev3*mod_boost,0,1.5))*(1-ads_oxyg)
    Wout = Wa*(1-0.18*clamp(sev3,0,1.5))*(1-0.55*clamp(ads_water,0,0.8))
    Vout = Va*(1-0.30*clamp(sev3,0,1.5))*(1-0.10*clamp(ads_oxyg,0,0.3))

    over = max(0.0, sev3-1.0)           # over-severity -> cracking -> yield loss
    Yout = Ya*(1-0.12*over)

    stability = 0.3 + 1.2*(O0-Oout) + 0.8*(W0-Wout)
    stability = clamp(stability, 0.0, 2.5)

    Q  = (0.8*sev3*Fkgph*10) + (0.02*(T3-300)*Fkgph)
    dP = 0.02*Fkgph*(P3/30.0) + 0.01*Fkgph*(1/max(WHSV,0.2))

    return np.array([Yout,Wout,Oout,Vout,stability,Q,dP], dtype=np.float32)

# -----------------------------
# 3) Generate dataset (POC scale)
# -----------------------------
rng = np.random.default_rng(7)
N = 2000
X = np.zeros((N,19), dtype=np.float32)
Y = np.zeros((N,7), dtype=np.float32)

for i in range(N):
    T1=rng.uniform(430,520); P1=rng.uniform(5,20);  a1=rng.uniform(0.1,1.0); tau1=rng.uniform(0.5,3.0)
    T2=rng.uniform(440,540); P2=rng.uniform(5,25);  a2=rng.uniform(0.1,1.0); tau2=rng.uniform(0.5,3.0)
    T3=rng.uniform(560,720); P3=rng.uniform(10,60); WHSV=rng.uniform(0.5,5.0); H2=rng.uniform(0.5,5.0)
    zfrac=rng.uniform(0.2,1.0); mload=rng.uniform(0.0,1.0); act=rng.uniform(0.6,1.0)
    Fkgph=rng.uniform(0.2,1.0)                  # around 10 kg/day ≈ 0.417 kg/h
    O0=rng.uniform(0.25,0.45); W0=rng.uniform(0.15,0.35); V0=rng.uniform(50,250)

    u=np.array([T1,P1,a1,tau1,T2,P2,a2,tau2,T3,P3,WHSV,H2,zfrac,mload,act,Fkgph,O0,W0,V0],dtype=np.float32)
    X[i]=u; Y[i]=truth(u)

# normalize inputs
X_mean, X_std = X.mean(axis=0), X.std(axis=0)+1e-6
Xn = (X - X_mean)/X_std

# build node features per sample (5 nodes × (type + dynamic))
DYN = 19
Fdim = BASE + DYN

def build_node_features(xrow):
    dyn = np.zeros((N_NODES,DYN), dtype=np.float32)
    dyn[0,-4:] = xrow[-4:]      # Feed: [Fkgph, O0, W0, V0]
    dyn[1,0:4] = xrow[0:4]      # R-100
    dyn[2,4:8] = xrow[4:8]      # R-200
    dyn[3,8:15]= xrow[8:15]     # R-300 (13X bed inputs)
    return np.concatenate([type_onehot, dyn], axis=1)

X_nodes = np.stack([build_node_features(Xn[i]) for i in range(N)], axis=0)  # (N,5,Fdim)

# train/test split
tr, te = train_test_split(np.arange(N), test_size=0.2, random_state=0)
Xtr = torch.tensor(X_nodes[tr], dtype=torch.float32)
Ytr = torch.tensor(Y[tr], dtype=torch.float32)
Xte = torch.tensor(X_nodes[te], dtype=torch.float32)
Yte = torch.tensor(Y[te], dtype=torch.float32)

# -----------------------------
# 4) Message-passing GNN model
# -----------------------------
class SimpleGNN(nn.Module):
    def __init__(self, in_dim, hid=96, out_dim=7):
        super().__init__()
        self.l1 = nn.Linear(in_dim, hid)
        self.l2 = nn.Linear(hid, hid)
        self.l3 = nn.Linear(hid, hid)
        self.out= nn.Linear(hid, out_dim)

    def forward(self, x_nodes, A_norm):
        # x_nodes: (B, N, F)
        h = F.relu(self.l1(x_nodes))
        h = torch.matmul(A_norm, h)      # message passing 1
        h = F.relu(self.l2(h))
        h = torch.matmul(A_norm, h)      # message passing 2
        h = F.relu(self.l3(h))
        hg = h.mean(dim=1)               # graph pooling
        return self.out(hg)

model = SimpleGNN(Fdim, hid=96, out_dim=7)
opt = torch.optim.Adam(model.parameters(), lr=2e-3, weight_decay=1e-4)

# training
batch = 128
for epoch in range(1, 26):
    model.train()
    order = torch.randperm(Xtr.shape[0])
    for i in range(0, Xtr.shape[0], batch):
        sel = order[i:i+batch]
        pred = model(Xtr[sel], An_t)
        loss = F.mse_loss(pred, Ytr[sel])
        opt.zero_grad(); loss.backward(); opt.step()

    if epoch % 5 == 0:
        model.eval()
        with torch.no_grad():
            P = model(Xte, An_t)
        rmse = torch.sqrt(torch.mean((P - Yte)**2, dim=0)).numpy()
        print(f"epoch {epoch:02d} test_RMSE:", np.round(rmse, 4))

# report KPIs
names = ["Yield","Moisture","Oxygenates","Viscosity","Stability","Q_kW","dP_bar"]
model.eval()
with torch.no_grad():
    P = model(Xte, An_t).numpy()
Yte_np = Yte.numpy()

for j,nm in enumerate(names):
    print(nm, "R2=", round(r2_score(Yte_np[:,j], P[:,j]), 4),
              "MAE=", round(mean_absolute_error(Yte_np[:,j], P[:,j]), 4))

# SCADA-style what-if: change only R-300 temperature (T3) and see outputs change
def predict_one(u_raw):
    x = (u_raw - X_mean)/X_std
    xnodes = torch.tensor(build_node_features(x)[None,:,:], dtype=torch.float32)
    with torch.no_grad():
        out = model(xnodes, An_t).numpy().ravel()
    return dict(zip(names, out))

u0 = X[0].copy()
base = predict_one(u0)
u1 = u0.copy(); u1[8] = 700.0   # T3 index in vector = 8
hot  = predict_one(u1)

print("\nBase:", base)
print("R-300 T up:", hot)


In [ ]:
import pandas as pd

# --- Table 6 (design variables) ---
design = pd.DataFrame({
    "Run": list(range(1, 16)),
    "SiO2_Al2O3": [50,80,80,80,50,80,23,50,23,50,23,23,50,50,50],
    "ZSM5_wt_pct": [22,30,22,13,30,22,22,22,13,30,30,22,13,13,22],
    "Temp_C": [500,500,450,500,450,550,550,500,500,550,500,450,550,450,500],
    "CTO": [5,6,5,4,6,5,5,5,4,6,6,5,4,4,5],
})

# --- Table 12 (measured responses) ---
resp = pd.DataFrame([
    [1,  73.73, 51.64, 4.14, 15.05, 23.28, 7.27, 14.82, 9.17],
    [2,  85.58, 64.78, 4.74, 19.21, 29.31, 8.34, 12.47, 11.52],
    [3,  71.53, 53.85, 2.42, 14.99, 25.30, 10.02, 7.66, 11.14],
    [4,  74.71, 56.30, 3.17, 15.20, 27.93, 10.63, 7.78, 10.00],
    [5,  59.08, 46.67, 2.93, 13.79, 21.73, 6.65, 5.77, 8.22],
    [6,  66.22, 53.87, 5.27, 18.00, 21.99, 6.33, 6.02, 8.61],
    [7,  70.84, 57.06, 6.57, 17.23, 23.70, 5.90, 7.87, 9.56],
    [8,  66.57, 40.77, 2.37, 12.05, 18.88, 7.63, 18.17, 7.47],
    [9,  63.59, 53.22, 5.67, 12.52, 23.47, 6.14, 4.23, 11.55],
    [10, 72.95, 52.09, 6.24, 12.48, 22.88, 6.66, 14.20, 10.50],
    [11, 72.02, 43.86, 5.47, 10.15, 19.04, 8.01, 20.14, 9.21],
    [12, 65.47, 43.52, 1.45, 11.33, 23.35, 10.27, 11.67, 7.39],
    [13, 73.08, 49.91, 5.36, 14.05, 21.83, 7.73, 15.44, 8.68],
    [14, 81.27, 50.21, 2.42, 12.84, 24.88, 12.54, 18.52, 10.08],
    [15, 70.50, 42.90, 2.77, 10.74, 21.80, 9.06, 18.54, 7.58],
], columns=[
    "Run","MassBalance_pct","Conversion_pct","DryGas_pct","LPG_pct",
    "GasolineNaphtha_pct","LCO_pct","BottomFrac_pct","Coke_pct"
])

df = design.merge(resp, on="Run")
df.to_csv("fcc_biooil_zeolite_dataset.csv", index=False)
print(df.head())


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[["SiO2_Al2O3","ZSM5_wt_pct","Temp_C","CTO"]].values.astype(np.float32)
Y = df[["Conversion_pct","GasolineNaphtha_pct","Coke_pct","DryGas_pct","BottomFrac_pct"]].values.astype(np.float32)

Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=0.25, random_state=7)

sx = StandardScaler().fit(Xtr)
sy = StandardScaler().fit(Ytr)

Xtr_s = sx.transform(Xtr); Xte_s = sx.transform(Xte)
Ytr_s = sy.transform(Ytr); Yte_s = sy.transform(Yte)

Xtr_t = torch.tensor(Xtr_s); Ytr_t = torch.tensor(Ytr_s)
Xte_t = torch.tensor(Xte_s); Yte_t = torch.tensor(Yte_s)

model = nn.Sequential(
    nn.Linear(4, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, Y.shape[1])
)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(1500):
    model.train()
    pred = model(Xtr_t)
    loss = loss_fn(pred, Ytr_t)
    opt.zero_grad(); loss.backward(); opt.step()

model.eval()
with torch.no_grad():
    te_pred = sy.inverse_transform(model(Xte_t).numpy())
print("Sample predictions (test):")
print(te_pred[:3])
